In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dataguruji/rankin-need-data/candidate_documents.parquet
/kaggle/input/datasets/dataguruji/rankin-need-data/behavior_features.parquet
/kaggle/input/datasets/dataguruji/rankin-need-data/candidate_ids.npy
/kaggle/input/datasets/dataguruji/rankin-need-data/bm25.pkl
/kaggle/input/datasets/dataguruji/rankin-need-data/jd_hybrid_profile.pkl
/kaggle/input/datasets/dataguruji/rankin-need-data/candidate_embeddings.npy
/kaggle/input/datasets/dataguruji/rankin-need-data/tfidf_vectorizer.joblib
/kaggle/input/datasets/dataguruji/rankin-need-data/integrity_features.parquet
/kaggle/input/datasets/dataguruji/rankin-need-data/tfidf_matrix.joblib
/kaggle/input/datasets/dataguruji/rankin-need-data/candidate_metadata.parquet
/kaggle/input/datasets/dataguruji/rankin-need-data/top3000_candidates.parquet


In [3]:
from tqdm.auto import tqdm
from collections import defaultdict

from sentence_transformers import SentenceTransformer


In [4]:
!pip install -q rank_bm25

In [5]:
from rank_bm25 import BM25Okapi


In [6]:
top_candidates = pd.read_parquet(
    "/kaggle/input/datasets/dataguruji/rankin-need-data/top3000_candidates.parquet"
)

print(top_candidates.shape)

(3000, 11)


In [7]:
behavior = pd.read_parquet(
    "/kaggle/input/datasets/dataguruji/rankin-need-data/behavior_features.parquet"
)

integrity = pd.read_parquet(
    "/kaggle/input/datasets/dataguruji/rankin-need-data/integrity_features.parquet"
)

metadata = pd.read_parquet(
    "/kaggle/input/datasets/dataguruji/rankin-need-data/candidate_metadata.parquet"
)

In [8]:
top_candidates = top_candidates.merge(

    behavior,

    on="candidate_id",

    how="left"

)

top_candidates = top_candidates.merge(

    integrity,

    on="candidate_id",

    how="left"

)

top_candidates = top_candidates.merge(

    metadata,

    on="candidate_id",

    how="left"

)

In [9]:
print(top_candidates.shape)

top_candidates.head()

(3000, 34)


,candidate_id,profile_document,career_document,skills_document,education_document,certification_document,retrieval_document,num_words,rrf_score,semantic_score,...,headline,current_title,current_company,industry,country,years_experience,highest_degree,num_jobs,num_skills,num_certifications
0,CAND_0098104,Java Developer | Full-stack development Softwa...,\nThe candidate currently works\nas a\n\nJava ...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.E.\n\nin\n\nComputer Science\...,,Java Developer | Full-stack development Softwa...,318,0.033333,0.488166,...,Java Developer | Full-stack development,Java Developer,Dunder Mifflin,Paper Products,India,1.9,B.E.,1,7,0
1,CAND_0019575,Frontend Engineer | Cloud & DevOps Software en...,\nThe candidate currently works\nas a\n\nFront...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nM.E.\n\nin\n\nComputer Science\...,,Frontend Engineer | Cloud & DevOps Software en...,328,0.032522,0.460256,...,Frontend Engineer | Cloud & DevOps,Frontend Engineer,Flipkart,E-commerce,India,2.1,M.E.,1,8,0
2,CAND_0017455,Full Stack Developer | Cloud & DevOps Software...,\nThe candidate currently works\nas a\n\nFull ...,\nThe candidate has\n\nbeginner\n\nproficiency...,\nCompleted\n\nB.Sc\n\nin\n\nComputer Engineer...,,Full Stack Developer | Cloud & DevOps Software...,320,0.032522,0.469220,...,Full Stack Developer | Cloud & DevOps,Full Stack Developer,Mindtree,IT Services,India,2.1,B.Sc,1,6,0
3,CAND_0062234,Cloud Engineer | Backend systems & APIs Softwa...,\nThe candidate currently works\nas a\n\nCloud...,\nThe candidate has\n\nbeginner\n\nproficiency...,\nCompleted\n\nB.Tech\n\nin\n\nComputer Engine...,,Cloud Engineer | Backend systems & APIs Softwa...,333,0.031250,0.474204,...,Cloud Engineer | Backend systems & APIs,Cloud Engineer,Wayne Enterprises,Conglomerate,India,1.3,B.Tech,1,7,0
4,CAND_0096716,QA Engineer | Full-stack development Software ...,\nThe candidate currently works\nas a\n\nQA En...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nM.E.\n\nin\n\nStatistics\n\nfro...,,QA Engineer | Full-stack development Software ...,331,0.030579,0.479830,...,QA Engineer | Full-stack development,QA Engineer,Razorpay,Fintech,India,1.5,M.E.,1,7,0


In [10]:
import numpy as np

def normalize(series):

    s = series.fillna(0).astype(float)

    mn = s.min()

    mx = s.max()

    if mx == mn:
        return np.ones(len(s))

    return (s - mn) / (mx - mn)

In [11]:
top_candidates["response_rate_norm"] = normalize(
    top_candidates["response_rate"]
)

top_candidates["github_norm"] = normalize(
    top_candidates["github_score"]
)

top_candidates["endorsement_norm"] = normalize(
    top_candidates["endorsements"]
)

top_candidates["views_norm"] = normalize(
    top_candidates["profile_views"]
)

top_candidates["search_norm"] = normalize(
    top_candidates["search_appearance"]
)

top_candidates["saved_norm"] = normalize(
    top_candidates["saved_by_recruiters"]
)

In [12]:
response = top_candidates["response_time"].fillna(999)

response = response.clip(
    upper=240
)

top_candidates["response_time_score"] = 1 - normalize(response)

In [13]:
top_candidates["open_score"] = (

top_candidates["open_to_work"]

.astype(float)

)

In [14]:
top_candidates["profile_score"] = normalize(

top_candidates["profile_complete"]

)

In [15]:
top_candidates["behavior_score"] = (

0.25*top_candidates["response_rate_norm"]

+

0.15*top_candidates["open_score"]

+

0.15*top_candidates["response_time_score"]

+

0.10*top_candidates["profile_score"]

+

0.10*top_candidates["github_norm"]

+

0.10*top_candidates["saved_norm"]

+

0.05*top_candidates["search_norm"]

+

0.05*top_candidates["views_norm"]

+

0.05*top_candidates["endorsement_norm"]

)

In [16]:
top_candidates[

[
"candidate_id",

"behavior_score"

]

].head()

,candidate_id,behavior_score
0,CAND_0098104,0.580133
1,CAND_0019575,0.401516
2,CAND_0017455,0.297645
3,CAND_0062234,0.384928
4,CAND_0096716,0.503586


In [17]:
top_candidates["integrity_score"] = (

top_candidates["integrity_score"]

.fillna(0)

)

In [18]:
top_candidates["integrity_penalty"] = (

1 -

top_candidates["integrity_score"]

)

In [19]:
top_candidates["integrity_rank_score"] = (

1 -

top_candidates["integrity_penalty"]

)

In [20]:
import re

class CareerEvidenceScorer:

    def __init__(self):

        self.production_titles = {

            "machine learning engineer":1.0,
            "ai engineer":1.0,
            "ml engineer":1.0,
            "software engineer":0.90,
            "backend engineer":0.85,
            "research engineer":0.90,
            "data scientist":0.80,
            "applied scientist":1.0,
            "search engineer":1.0,
            "recommendation engineer":1.0,
            "nlp engineer":1.0

        }

        self.production_keywords={

            "retrieval",
            "ranking",
            "search",
            "recommendation",
            "recommendation system",
            "vector search",
            "faiss",
            "pinecone",
            "milvus",
            "rag",
            "embedding",
            "mlops",
            "production",
            "deployment",
            "serving",
            "distributed",
            "kubernetes",
            "docker"

        }

In [21]:
def title_score(self,text):

    text=str(text).lower()

    score=0

    for role,val in self.production_titles.items():

        if role in text:

            score=max(score,val)

    return score

In [22]:
def keyword_score(self,text):

    text=str(text).lower()

    hits=0

    for kw in self.production_keywords:

        if kw in text:

            hits+=1

    return min(hits/10,1.0)

In [23]:
def experience_score(self,years):

    try:

        years=float(years)

    except:

        return 0

    if years>=7:

        return 1

    if years>=5:

        return .9

    if years>=3:

        return .7

    return .3

In [24]:
def score(self,row):

    profile=row["profile_document"]

    career=row["career_document"]

    years=row["years_experience"]

    title=self.title_score(profile)

    keyword=self.keyword_score(career)

    exp=self.experience_score(years)

    return (

        0.45*title+

        0.35*keyword+

        0.20*exp

    )

In [25]:
CareerEvidenceScorer.title_score=title_score
CareerEvidenceScorer.keyword_score=keyword_score
CareerEvidenceScorer.experience_score=experience_score
CareerEvidenceScorer.score=score

In [26]:
engine=CareerEvidenceScorer()

top_candidates["career_score"]=top_candidates.apply(

engine.score,

axis=1

)

In [27]:
top_candidates[
[
"candidate_id",
"career_score"
]
].head()

,candidate_id,career_score
0,CAND_0098104,0.465
1,CAND_0019575,0.465
2,CAND_0017455,0.535
3,CAND_0062234,0.535
4,CAND_0096716,0.570


In [28]:
# ============================================================
# NOTEBOOK 4
# Redrob AI Hiring Challenge
# Part 1 : Loading Artifacts + Data Preparation
# ============================================================

import os
import gc
import re
import ast
import math
import pickle
import warnings

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns",None)
pd.set_option("display.width",150)

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/dataguruji/rankin-need-data")

TOPK = 3000

In [29]:
# ============================================================
# Load Notebook 3 Artifacts
# ============================================================

candidate_df = pd.read_parquet(
    DATA_DIR/"top3000_candidates.parquet"
)

metadata_df = pd.read_parquet(
    DATA_DIR/"candidate_metadata.parquet"
)

candidate_embeddings = np.load(
    DATA_DIR/"candidate_embeddings.npy"
)

candidate_ids = np.load(
    DATA_DIR/"candidate_ids.npy",
    allow_pickle=True
)

print(candidate_df.shape)
print(metadata_df.shape)
print(candidate_embeddings.shape)
print(candidate_ids.shape)

(3000, 11)
(100000, 11)
(100000, 384)
(100000,)


In [30]:
# ============================================================
# Load Hybrid JD
# ============================================================

with open(DATA_DIR/"jd_hybrid_profile.pkl","rb") as f:
    jd_profile = pickle.load(f)

print(type(jd_profile))

<class 'dict'>


In [31]:
# ============================================================
# Merge Metadata
# ============================================================

candidate_df = candidate_df.merge(
    metadata_df,
    on="candidate_id",
    how="left"
)

print(candidate_df.shape)

(3000, 21)


In [32]:
# ============================================================
# Fill Missing Values
# ============================================================

TEXT_COLUMNS = [

"profile_document",

"career_document",

"skills_document",

"education_document",

"certification_document",

"retrieval_document"

]

for c in TEXT_COLUMNS:

    candidate_df[c] = (
        candidate_df[c]
        .fillna("")
        .astype(str)
    )

candidate_df.fillna(0,inplace=True)

In [33]:
# ============================================================
# Normalize Scores
# ============================================================

scaler = MinMaxScaler()

candidate_df["semantic_score_norm"] = scaler.fit_transform(
    candidate_df[["semantic_score"]]
)

candidate_df["retrieval_score_norm"] = scaler.fit_transform(
    candidate_df[["retrieval_score"]]
)

candidate_df["rrf_score_norm"] = scaler.fit_transform(
    candidate_df[["rrf_score"]]
)

In [34]:
# ============================================================
# Text Cleaning
# ============================================================

def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r"\n"," ",text)

    text = re.sub(r"\s+"," ",text)

    return text.strip()

In [35]:
for c in TEXT_COLUMNS:

    candidate_df[c] = candidate_df[c].apply(clean_text)

In [36]:
# ============================================================
# Build Complete Resume Text
# ============================================================

candidate_df["resume_text"] = (

candidate_df["profile_document"]

+" "

+candidate_df["career_document"]

+" "

+candidate_df["skills_document"]

+" "

+candidate_df["education_document"]

+" "

+candidate_df["certification_document"]

)

In [37]:
candidate_df["resume_length"] = (
    candidate_df["resume_text"]
    .str.split()
    .apply(len)
)

In [38]:
# ============================================================
# JD Parsing Utilities
# ============================================================

def object_to_text(obj):

    if isinstance(obj,str):
        return obj

    if isinstance(obj,list):
        return " ".join(map(str,obj))

    if isinstance(obj,dict):
        return " ".join(
            map(str,obj.values())
        )

    return str(obj)

In [39]:
jd_text = object_to_text(jd_profile)
jd_text = clean_text(jd_text)

print(jd_text[:1000])

job description: senior ai engineer — founding team company: redrob ai (series a ai-native talent intelligence platform) location: pune/noida, india (hybrid — flexible cadence) | open to relocation candidates from tier-1 indian cities employment type: full-time experience required: 5–9 years (see "what we mean by this" below) let's be honest about this role we're going to write this jd differently from most. we're a series a company that just raised our round and we're building a new ai engineering org from scratch. this is the kind of role where the jd changes every six months because the company changes every six months. so instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before. if you've spent your career at google or meta and you want a well-scoped role with a defined ladder, this isn't it. if you've spent your career bouncing between early-stage startups and you want to "just code" without having to think a

In [40]:
# ============================================================
# Keyword Extraction
# ============================================================

WORD_RE = re.compile(r"[a-zA-Z][a-zA-Z0-9+#.]+")

def tokenize(text):

    return WORD_RE.findall(text.lower())

JD_WORDS = set(tokenize(jd_text))

print(len(JD_WORDS))

742


In [41]:
candidate_df["resume_words"] = (
    candidate_df["resume_text"]
    .apply(tokenize)
)

In [42]:
candidate_df["resume_word_set"] = (
    candidate_df["resume_words"]
    .apply(set)
)

In [43]:
# ============================================================
# JD Coverage
# ============================================================

def jd_overlap(words):

    if len(words)==0:

        return 0

    return len(words & JD_WORDS)/len(JD_WORDS)

In [44]:
candidate_df["jd_overlap"] = (
    candidate_df["resume_word_set"]
    .apply(jd_overlap)
)

In [45]:
# ============================================================
# Resume Statistics
# ============================================================

candidate_df["num_unique_words"] = (

candidate_df["resume_word_set"]

.apply(len)

)

candidate_df["lexical_diversity"] = (

candidate_df["num_unique_words"]

/

candidate_df["resume_length"]

)

candidate_df["lexical_diversity"] = (

candidate_df["lexical_diversity"]

.fillna(0)

)

In [46]:
# ============================================================
# Initial Base Score
# ============================================================

candidate_df["base_score"] = (

0.50*candidate_df.semantic_score_norm

+

0.30*candidate_df.rrf_score_norm

+

0.20*candidate_df.retrieval_score_norm

)

In [47]:
candidate_df = candidate_df.sort_values(
    "base_score",
    ascending=False
).reset_index(drop=True)

In [48]:
print(candidate_df.head())

   candidate_id                                   profile_document                                    career_document  \
0  CAND_0098104  java developer | full-stack development softwa...  the candidate currently works as a java develo...   
1  CAND_0067410  content writer | ai enthusiast | building with...  the candidate currently works as a content wri...   
2  CAND_0062923  content writer | exploring ai & genai applicat...  the candidate currently works as a content wri...   
3  CAND_0096716  qa engineer | full-stack development software ...  the candidate currently works as a qa engineer...   
4  CAND_0067486  full stack developer | full-stack development ...  the candidate currently works as a full stack ...   

                                     skills_document                                 education_document certification_document  \
0  the candidate has intermediate proficiency in ...  completed b.e. in computer science from region...                          
1  the candid

In [49]:
row = candidate_df.iloc[0]

print(row["career_document"])
print(row["skills_document"])
print(row["education_document"])
print(row["certification_document"])

the candidate currently works as a java developer at dunder mifflin for 22 months. industry: paper products. responsibilities: frontend engineering at a media company. react, typescript, and the typical surrounding tooling (webpack, jest, cypress). built the company's design system from scratch and led the migration from a legacy angularjs app. strong on the frontend craft — accessibility, performance, animations — but limited backend exposure.
the candidate has intermediate proficiency in django with 11 months of experience and 1 endorsements. the candidate has intermediate proficiency in redux with 35 months of experience and 2 endorsements. the candidate has beginner proficiency in bigquery with 4 months of experience and 10 endorsements. the candidate has intermediate proficiency in object detection with 18 months of experience and 7 endorsements. the candidate has intermediate proficiency in photoshop with 14 months of experience and 5 endorsements. the candidate has beginner prof

In [50]:
# ============================================================
# PART 2A
# Career Evidence Parsing
# ============================================================

import re
import numpy as np

# ----------------------------
# Proficiency Mapping
# ----------------------------

PROFICIENCY_MAP = {
    "beginner": 1,
    "intermediate": 2,
    "advanced": 3,
    "expert": 4
}

DEGREE_SCORE = {
    "ph.d":1.00,
    "phd":1.00,
    "doctorate":1.00,
    "m.tech":0.95,
    "m.e.":0.95,
    "m.s":0.95,
    "mba":0.90,
    "b.tech":0.85,
    "b.e.":0.85,
    "b.sc":0.80,
    "bca":0.75,
    "diploma":0.60
}

TIER_SCORE = {
    "tier_1":1.00,
    "tier_2":0.90,
    "tier_3":0.80,
    "tier_4":0.70,
    "unknown":0.60
}

In [51]:
# ============================================================
# Parse Career Document
# ============================================================

career_pattern = re.compile(
    r"works as a (.*?) at (.*?) for (\d+) months",
    re.I
)

industry_pattern = re.compile(
    r"industry:\s*(.*?)\.",
    re.I
)

responsibility_pattern = re.compile(
    r"responsibilities:\s*(.*)",
    re.I
)

def parse_career(text):

    role = ""
    company = ""
    industry = ""
    months = 0
    responsibility = ""

    m = career_pattern.search(text)

    if m:
        role = m.group(1).strip()
        company = m.group(2).strip()
        months = int(m.group(3))

    m = industry_pattern.search(text)

    if m:
        industry = m.group(1).strip()

    m = responsibility_pattern.search(text)

    if m:
        responsibility = m.group(1).strip()

    return pd.Series([
        role,
        company,
        industry,
        months,
        responsibility
    ])

In [52]:
candidate_df[
[
"current_role",
"current_company",
"industry",
"career_months",
"responsibility_text"
]
] = candidate_df["career_document"].apply(parse_career)

In [53]:
# ============================================================
# Parse Skills
# ============================================================

skill_pattern = re.compile(

r"proficiency in (.*?) with (\d+) months of experience and (\d+) endorsements",

re.I
)

def parse_skills(text):

    skills=[]

    for m in skill_pattern.finditer(text):

        skills.append({

            "skill":m.group(1).strip().lower(),

            "months":int(m.group(2)),

            "endorsements":int(m.group(3)),

            "proficiency":PROFICIENCY_MAP.get(

                text[m.start():m.start()+40].split()[4].lower(),

                1

            )

        })

    return skills

In [54]:
candidate_df["parsed_skills"] = (
    candidate_df["skills_document"]
    .apply(parse_skills)
)

In [55]:
# ============================================================
# Aggregate Skill Statistics
# ============================================================

def aggregate_skill_features(skill_list):

    if len(skill_list)==0:

        return pd.Series([
            0,
            0,
            0,
            0,
            0
        ])

    n=len(skill_list)

    total_months=sum(s["months"] for s in skill_list)

    total_endorse=sum(s["endorsements"] for s in skill_list)

    avg_prof=np.mean([

        s["proficiency"]

        for s in skill_list

    ])

    max_prof=max(

        s["proficiency"]

        for s in skill_list

    )

    return pd.Series([

        n,

        total_months,

        total_endorse,

        avg_prof,

        max_prof

    ])

In [56]:
candidate_df[
[
"num_skills",
"skill_months",
"endorsement_count",
"avg_proficiency",
"max_proficiency"
]
]=candidate_df["parsed_skills"].apply(
aggregate_skill_features
)

In [57]:
# ============================================================
# Parse Education
# ============================================================

degree_pattern = re.compile(

r"completed (.*?) in",

re.I
)

tier_pattern = re.compile(

r"institution tier:\s*(tier_\d)",

re.I
)

cgpa_pattern = re.compile(

r"([0-9]\.[0-9]+)\s*cgpa",

re.I
)

def parse_education(text):

    degree=""

    tier="unknown"

    cgpa=0

    m=degree_pattern.search(text)

    if m:

        degree=m.group(1).strip().lower()

    m=tier_pattern.search(text)

    if m:

        tier=m.group(1).lower()

    m=cgpa_pattern.search(text)

    if m:

        cgpa=float(m.group(1))

    return pd.Series([

        degree,

        tier,

        cgpa

    ])

In [58]:
candidate_df[
[
"degree",
"institution_tier",
"cgpa"
]
]=candidate_df[
"education_document"
].apply(parse_education)

In [59]:
# ============================================================
# Parse Certifications
# ============================================================

def certification_count(text):

    if len(text.strip())==0:

        return 0

    return len(

        re.findall(r"\.",text)

    )

candidate_df["num_certifications"]=(
candidate_df[
"certification_document"
].apply(certification_count)
)

In [60]:
# ============================================================
# Degree / Tier Scores
# ============================================================

candidate_df["degree_score"]=candidate_df[
"degree"
].map(DEGREE_SCORE).fillna(0.70)

candidate_df["tier_score"]=candidate_df[
"institution_tier"
].map(TIER_SCORE).fillna(0.60)

candidate_df["cgpa_score"]=(
candidate_df["cgpa"]/10
).clip(0,1)

In [61]:
# ============================================================
# Experience Score
# ============================================================

candidate_df["experience_score"]=(
candidate_df["career_months"]/120
).clip(0,1)

In [62]:
# ============================================================
# Skill Strength
# ============================================================

candidate_df["skill_strength"]=(
0.40*(candidate_df["avg_proficiency"]/4)
+
0.30*(candidate_df["endorsement_count"]/50).clip(0,1)
+
0.30*(candidate_df["skill_months"]/300).clip(0,1)
)

In [63]:
# ============================================================
# Education Strength
# ============================================================

candidate_df["education_strength"]=(
0.45*candidate_df["degree_score"]
+
0.25*candidate_df["tier_score"]
+
0.30*candidate_df["cgpa_score"]
)

In [64]:
# ============================================================
# Career Evidence Score
# ============================================================

candidate_df["career_evidence_score"]=(
0.35*candidate_df["experience_score"]
+
0.40*candidate_df["skill_strength"]
+
0.20*candidate_df["education_strength"]
+
0.05*(candidate_df["num_certifications"]/10).clip(0,1)
)

In [65]:
print(candidate_df[
[
"candidate_id",
"career_months",
"num_skills",
"endorsement_count",
"degree",
"career_evidence_score"
]
].head())

   candidate_id  career_months  num_skills  endorsement_count degree  career_evidence_score
0  CAND_0098104             22         7.0               35.0   b.e.               0.386787
1  CAND_0067410             49        12.0               27.0   m.s.               0.468877
2  CAND_0062923             22        12.0               60.0   ph.d               0.481367
3  CAND_0096716             18         7.0              111.0   m.e.               0.435980
4  CAND_0067486             12         7.0               67.0   m.e.               0.411360


In [66]:
# ============================================================
# PART 2B.1
# Structured JD Feature Extraction
# ============================================================

from collections import Counter
import pandas as pd
import numpy as np
import re

In [67]:
# ------------------------------------------------------------
# Parse JD Profile
# ------------------------------------------------------------

# jd_profile was already loaded in Part 1

print(type(jd_profile))

<class 'dict'>


In [68]:
# ------------------------------------------------------------
# Extract Components
# ------------------------------------------------------------

JD_TEXT = ""

JD_KEYWORDS = pd.DataFrame()

JD_IMPORTANT = pd.DataFrame()

JD_NOUN_CHUNKS = []

JD_ENTITIES = []

if isinstance(jd_profile, dict):

    JD_TEXT = jd_profile.get("raw_text","")

    JD_KEYWORDS = jd_profile.get("keywords",pd.DataFrame())

    JD_IMPORTANT = jd_profile.get("important_sentences",pd.DataFrame())

    JD_NOUN_CHUNKS = jd_profile.get("noun_chunks",[])

    JD_ENTITIES = jd_profile.get("entities",[])

In [69]:
print("JD length:",len(JD_TEXT))

print("Keywords:",len(JD_KEYWORDS))

print("Important:",len(JD_IMPORTANT))

print("Chunks:",len(JD_NOUN_CHUNKS))

print("Entities:",len(JD_ENTITIES))

JD length: 9564
Keywords: 50
Important: 20
Chunks: 298
Entities: 88


In [70]:
# ------------------------------------------------------------
# Build Keyword Weight Dictionary
# ------------------------------------------------------------

keyword_weights = {}

if len(JD_KEYWORDS)>0:

    for _,row in JD_KEYWORDS.iterrows():

        keyword = str(row["Keyword"]).lower()

        score = float(row["Score"])

        keyword_weights[keyword]=score

In [71]:
print(list(keyword_weights.items())[:10])

[('indian cities employment', 0.0011233067354827176), ('cities employment type', 0.0011233067354827176), ('full-time experience required', 0.0015361162600894278), ('employment type', 0.004653050628870325), ('founding team company', 0.007881629296197038), ('open to relocation', 0.01604039409578463), ('indian cities', 0.01604039409578463), ('cities employment', 0.0163554979383432), ('full-time experience', 0.022244310483324082), ('experience required', 0.022244310483324082)]


In [72]:
# ------------------------------------------------------------
# Important Sentence List
# ------------------------------------------------------------

important_sentences=[]

if len(JD_IMPORTANT)>0:

    important_sentences=list(
        JD_IMPORTANT["Sentence"]
    )

In [73]:
print("Important Sentences:",len(important_sentences))

Important Sentences: 20


In [74]:
# ------------------------------------------------------------
# Normalize Noun Chunks
# ------------------------------------------------------------

noun_chunks=[]

for chunk in JD_NOUN_CHUNKS:

    chunk=str(chunk).lower()

    chunk=re.sub(r"\s+"," ",chunk)

    noun_chunks.append(chunk)

In [75]:
noun_chunks=list(set(noun_chunks))

print(len(noun_chunks))

298


In [76]:
# ------------------------------------------------------------
# Normalize Entities
# ------------------------------------------------------------

entity_names=[]

for ent in JD_ENTITIES:

    if isinstance(ent,dict):

        entity_names.append(

            ent["Entity"].lower()

        )

entity_names=list(set(entity_names))

In [77]:
print(entity_names[:30])

['xgboost', 'meta', 'weeks 9-12', 'ai engineering', 'redrob', 'india', 'hyderabad', 'the intelligent candidate discovery & ranking challenge', 'elasticsearch', '100', '5+ years', 'tcs', 'milvus', 'first 90 days', 'ml/ai', '6 months', '1000', 'a week', 'pune/noida', '6-8 years', '4 years', 'at least one', 'jd', 'openai', '20', 'bge', 'lora', 'the next year', 'weeks 1-3: audit', 'the last 18 months']


In [78]:
# ------------------------------------------------------------
# Resume Token Dictionary
# ------------------------------------------------------------

candidate_df["resume_counter"]=(
candidate_df["resume_text"]
.apply(
lambda x: Counter(tokenize(x))
)
)

In [79]:
print(candidate_df.resume_counter.iloc[0].most_common(20))

[('the', 19), ('and', 13), ('in', 11), ('with', 10), ('candidate', 10), ('of', 9), ('experience', 8), ('has', 8), ('proficiency', 7), ('months', 7), ('endorsements.', 7), ('intermediate', 5), ('at', 4), ('java', 3), ('developer', 3), ('software', 3), ('backend', 3), ('my', 3), ('is', 3), ('but', 3)]


In [80]:
# ============================================================
# PART 2B.2
# Weighted Keyword Matching
# ============================================================

TOTAL_KEYWORD_WEIGHT = sum(keyword_weights.values())

print("Total keyword weight:", TOTAL_KEYWORD_WEIGHT)

Total keyword weight: 2.7177959383040284


In [81]:
# ------------------------------------------------------------
# Resume Tokens
# ------------------------------------------------------------

candidate_df["resume_tokens"] = (
    candidate_df["resume_text"]
    .apply(tokenize)
)

In [82]:
# ------------------------------------------------------------
# Weighted Keyword Match
# ------------------------------------------------------------

def weighted_keyword_score(tokens):

    tokens = set(tokens)

    score = 0.0

    for keyword, weight in keyword_weights.items():

        keyword_words = keyword.split()

        if len(keyword_words) == 1:

            if keyword in tokens:
                score += weight

        else:

            resume_text_lower = " ".join(tokens)

            for keyword, weight in keyword_weights.items():
                if keyword.lower() in resume_text_lower:
                    score += weight

    return score

In [83]:
candidate_df["weighted_keyword_score"] = (
    candidate_df["resume_tokens"]
    .apply(weighted_keyword_score)
)

In [84]:
candidate_df["weighted_keyword_score"] /= max(
    TOTAL_KEYWORD_WEIGHT,
    1e-9
)

In [85]:
candidate_df["weighted_keyword_score"] = (
    candidate_df["weighted_keyword_score"]
    .clip(0,1)
)

In [86]:
def keyword_coverage(tokens):

    tokens = set(tokens)

    matched = 0

    total = len(keyword_weights)

    for keyword in keyword_weights:

        if len(keyword.split()) == 1:

            if keyword in tokens:
                matched += 1

        else:

            if keyword in " ".join(tokens):
                matched += 1

    return matched / max(total,1)

In [87]:
candidate_df["keyword_coverage"] = (
    candidate_df["resume_tokens"]
    .apply(keyword_coverage)
)

In [88]:
IMPORTANT_THRESHOLD = (
    JD_KEYWORDS.Score.quantile(0.75)
)

important_keywords = {

    row.Keyword.lower()

    for _, row in JD_KEYWORDS.iterrows()

    if row.Score >= IMPORTANT_THRESHOLD

}

print(len(important_keywords))

13


In [89]:
def important_keyword_hits(tokens):

    tokens = set(tokens)

    hits = 0

    for keyword in important_keywords:

        if len(keyword.split()) == 1:

            if keyword in tokens:
                hits += 1

        else:

            if keyword in " ".join(tokens):
                hits += 1

    return hits

In [90]:
candidate_df["important_keyword_hits"] = (
    candidate_df["resume_tokens"]
    .apply(important_keyword_hits)
)

In [91]:
candidate_df["important_keyword_ratio"] = (
    candidate_df["important_keyword_hits"]
    /
    max(len(important_keywords),1)
)

In [92]:
def missing_keywords(tokens):

    tokens = set(tokens)

    missing = 0

    for keyword in important_keywords:

        if len(keyword.split()) == 1:

            if keyword not in tokens:
                missing += 1

        else:

            if keyword not in " ".join(tokens):
                missing += 1

    return missing

In [93]:
candidate_df["missing_important_keywords"] = (
    candidate_df["resume_tokens"]
    .apply(missing_keywords)
)

In [94]:
candidate_df["missing_keyword_penalty"] = (
    candidate_df["missing_important_keywords"]
    /
    max(len(important_keywords),1)
)

In [95]:
# ------------------------------------------------------------
# Resume Statistics (Required for Keyword Density)
# ------------------------------------------------------------

if "word_count" not in candidate_df.columns:
    candidate_df["word_count"] = (
        candidate_df["resume_text"]
        .str.split()
        .apply(len)
    )

if "char_count" not in candidate_df.columns:
    candidate_df["char_count"] = (
        candidate_df["resume_text"]
        .str.len()
    )

candidate_df["word_count"] = candidate_df["word_count"].clip(lower=1)

In [96]:
candidate_df["keyword_density"] = (
    candidate_df["important_keyword_hits"]
    /
    candidate_df["word_count"].clip(lower=1)
)

In [97]:
candidate_df["jd_keyword_feature"] = (

      0.45*candidate_df["weighted_keyword_score"]

    + 0.25*candidate_df["important_keyword_ratio"]

    + 0.20*candidate_df["keyword_coverage"]

    - 0.10*candidate_df["missing_keyword_penalty"]

)

In [98]:
candidate_df["jd_keyword_feature"] = (
    candidate_df["jd_keyword_feature"]
    .clip(0,1)
)

In [99]:
candidate_df[
[
    "candidate_id",
    "weighted_keyword_score",
    "keyword_coverage",
    "important_keyword_hits",
    "jd_keyword_feature"
]
].head()

,candidate_id,weighted_keyword_score,keyword_coverage,important_keyword_hits,jd_keyword_feature
0,CAND_0098104,1.0,0.16,3,0.462769
1,CAND_0067410,1.0,0.18,3,0.466769
2,CAND_0062923,1.0,0.16,3,0.462769
3,CAND_0096716,1.0,0.16,4,0.489692
4,CAND_0067486,1.0,0.16,4,0.489692


In [100]:
# ============================================================
# PART 2B.3 (Optimized)
# Semantic JD Alignment using Existing Candidate Embeddings
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

In [102]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [103]:
# ------------------------------------------------------------
# JD Embedding
# ------------------------------------------------------------

jd_embedding = embedding_model.encode(

    retrieval_document := JD_TEXT,

    normalize_embeddings=True,

    convert_to_numpy=True

).reshape(1, -1)

print(jd_embedding.shape)

(1, 384)


In [108]:
# ============================================================
# Build Candidate ID -> Embedding Index
# ============================================================

candidate_index = {
    cid: idx
    for idx, cid in enumerate(candidate_ids)
}

indices = [
    candidate_index[cid]
    for cid in candidate_df["candidate_id"]
]

top_candidate_embeddings = candidate_embeddings[indices]

print(top_candidate_embeddings.shape)

(3000, 384)


In [116]:
# ------------------------------------------------------------
# Cosine Similarity
# ------------------------------------------------------------
semantic_alignment = cosine_similarity(
    top_candidate_embeddings,
    jd_embedding
).flatten()

candidate_df["semantic_alignment"] = semantic_alignment

In [118]:
from sklearn.preprocessing import MinMaxScaler

candidate_df[["semantic_alignment"]] = (

    MinMaxScaler()

    .fit_transform(

        candidate_df[["semantic_alignment"]]

    )

)

In [119]:
candidate_df[
[
    "candidate_id",
    "semantic_alignment"
]
].head()

,candidate_id,semantic_alignment
0,CAND_0098104,0.406274
1,CAND_0067410,0.555443
2,CAND_0062923,0.613272
3,CAND_0096716,0.499133
4,CAND_0067486,0.216300


In [121]:
FEATURE_COLUMNS = [
    "semantic_score",
    "retrieval_score",
    "rrf_score",
    "career_evidence_score",
    "skill_strength",
    "education_strength",
    "experience_score",
    "resume_quality",
    "semantic_alignment",
    "jd_keyword_feature",
    "matched_skill_ratio",
    "resume_completeness",
    "lexical_diversity",
    "endorsement_density",
    "skill_density",
    "experience_per_skill"
]

In [123]:
FEATURE_WEIGHTS = {

    "semantic_score": 0.25,
    "semantic_alignment": 0.20,
    "retrieval_score": 0.12,
    "rrf_score": 0.05,

    "career_evidence_score": 0.10,
    "jd_keyword_feature": 0.08,

    "resume_quality": 0.05,
    "skill_strength": 0.05,
    "experience_score": 0.04,
    "education_strength": 0.03,

    "matched_skill_ratio": 0.01,

    "resume_completeness": 0.005,
    "lexical_diversity": 0.005,
    "endorsement_density": 0.005,
    "skill_density": 0.005,

    "experience_per_skill": 0.00
}

print(sum(FEATURE_WEIGHTS.values()))

1.0


In [124]:
candidate_df["ranking_score"] = 0.0

for feature, weight in FEATURE_WEIGHTS.items():
    if feature in candidate_df.columns:
        candidate_df["ranking_score"] += (
            candidate_df[feature] * weight
        )

candidate_df = (
    candidate_df
    .sort_values("ranking_score", ascending=False)
    .reset_index(drop=True)
)

candidate_df["rank"] = np.arange(1, len(candidate_df) + 1)

In [127]:
# ============================================================
# Role Suitability Engine
# ============================================================

GOOD_ROLES = {

    "ai engineer",

    "machine learning engineer",

    "ml engineer",

    "nlp engineer",

    "search engineer",

    "recommendation systems engineer",

    "applied ml engineer",

    "ai research engineer",

    "backend engineer",

    "software engineer",

    "data scientist",

    "senior data scientist"

}

In [128]:
BAD_ROLES = {

    "content writer",

    "customer support",

    "operations manager",

    "marketing manager",

    "sales executive",

    "mechanical engineer",

    "civil engineer",

    "electrical engineer",

    "graphic designer"

}

In [129]:
def role_score(role):

    role = str(role).lower()

    for good in GOOD_ROLES:
        if good in role:
            return 1.0

    for bad in BAD_ROLES:
        if bad in role:
            return 0.0

    return 0.5

In [130]:
candidate_df["role_score"] = (
    candidate_df["current_role"]
    .apply(role_score)
)

In [131]:
FEATURE_WEIGHTS = {

    "semantic_score":0.20,

    "semantic_alignment":0.20,

    "retrieval_score":0.10,

    "rrf_score":0.05,

    "career_evidence_score":0.10,

    "jd_keyword_feature":0.08,

    "resume_quality":0.05,

    "role_score":0.10,

    "skill_strength":0.05,

    "experience_score":0.03,

    "education_strength":0.02,

    "matched_skill_ratio":0.01,

    "resume_completeness":0.005,

    "lexical_diversity":0.005,

    "endorsement_density":0.005,

    "skill_density":0.005,

    "experience_per_skill":0.005
}

In [132]:
candidate_df["ranking_score"] = 0.0

for feature, weight in FEATURE_WEIGHTS.items():
    if feature in candidate_df.columns:
        candidate_df["ranking_score"] += (
            candidate_df[feature] * weight
        )

candidate_df = (
    candidate_df
    .sort_values("ranking_score", ascending=False)
    .reset_index(drop=True)
)

candidate_df["rank"] = np.arange(1, len(candidate_df) + 1)

In [133]:
candidate_df[
[
    "rank",
    "candidate_id",
    "ranking_score",
    "semantic_alignment",
    "semantic_score",
    "career_evidence_score",
    "current_role",
    "career_months"
]
].head(20)

,rank,candidate_id,ranking_score,semantic_alignment,semantic_score,career_evidence_score,current_role,career_months
0,1,CAND_0096172,0.581415,0.822298,0.567013,0.592167,nlp engineer,40
1,2,CAND_0091909,0.580416,0.808947,0.546463,0.624690,machine learning engineer,51
2,3,CAND_0091534,0.579958,0.825820,0.554467,0.623807,ai engineer,52
3,4,CAND_0036184,0.574663,0.813117,0.556181,0.581067,recommendation systems engineer,52
4,5,CAND_0020708,0.574469,0.820251,0.558412,0.602853,search engineer,50
5,6,CAND_0075249,0.572177,0.801160,0.557035,0.574360,applied ml engineer,36
6,7,CAND_0092706,0.570203,0.837574,0.517855,0.593733,ai research engineer,44
7,8,CAND_0089552,0.568493,0.807199,0.531897,0.596293,machine learning engineer,50
8,9,CAND_0078042,0.568284,0.809331,0.546244,0.571493,applied ml engineer,38
9,10,CAND_0069773,0.567147,0.852548,0.509429,0.609313,ai research engineer,50


In [135]:
candidate_df["raw_score"] = 0.0

for feature, weight in FEATURE_WEIGHTS.items():
    if feature in candidate_df.columns:
        candidate_df["raw_score"] += candidate_df[feature] * weight

In [137]:
candidate_df = (
    candidate_df
    .sort_values("raw_score", ascending=False)
    .reset_index(drop=True)
)

candidate_df["rank"] = np.arange(1, len(candidate_df) + 1)

In [140]:
candidate_df[
[
    "rank",
    "candidate_id",
    "raw_score",
    "confidence_score",
    "current_role"
]
].head(20)

,rank,candidate_id,raw_score,confidence_score,current_role
0,1,CAND_0096172,0.581415,98.00,nlp engineer
1,2,CAND_0091909,0.580416,97.76,machine learning engineer
2,3,CAND_0091534,0.579958,97.65,ai engineer
3,4,CAND_0036184,0.574663,96.38,recommendation systems engineer
4,5,CAND_0020708,0.574469,96.33,search engineer
5,6,CAND_0075249,0.572177,95.78,applied ml engineer
6,7,CAND_0092706,0.570203,95.31,ai research engineer
7,8,CAND_0089552,0.568493,94.90,machine learning engineer
8,9,CAND_0078042,0.568284,94.85,applied ml engineer
9,10,CAND_0069773,0.567147,94.57,ai research engineer


In [141]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(60, 88))

candidate_df["ranking_score"] = scaler.fit_transform(
    candidate_df[["raw_score"]]
).round(2)

In [145]:
candidate_df[
[
    "rank",
    "candidate_id",
    "ranking_score",
    "raw_score",
    "semantic_alignment",
    "semantic_score",
    "career_evidence_score",
    "current_role"
]
].head(20)

,rank,candidate_id,ranking_score,raw_score,semantic_alignment,semantic_score,career_evidence_score,current_role
0,1,CAND_0096172,90.00,0.581415,0.822298,0.567013,0.592167,nlp engineer
1,2,CAND_0091909,89.86,0.580416,0.808947,0.546463,0.624690,machine learning engineer
2,3,CAND_0091534,89.80,0.579958,0.825820,0.554467,0.623807,ai engineer
3,4,CAND_0036184,89.06,0.574663,0.813117,0.556181,0.581067,recommendation systems engineer
4,5,CAND_0020708,89.03,0.574469,0.820251,0.558412,0.602853,search engineer
5,6,CAND_0075249,88.71,0.572177,0.801160,0.557035,0.574360,applied ml engineer
6,7,CAND_0092706,88.43,0.570203,0.837574,0.517855,0.593733,ai research engineer
7,8,CAND_0089552,88.20,0.568493,0.807199,0.531897,0.596293,machine learning engineer
8,9,CAND_0078042,88.17,0.568284,0.809331,0.546244,0.571493,applied ml engineer
9,10,CAND_0069773,88.01,0.567147,0.852548,0.509429,0.609313,ai research engineer


In [146]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Scale raw scores to [0, 1]
scaled = MinMaxScaler().fit_transform(
    candidate_df[["raw_score"]]
).flatten()

# Stretch the top end
candidate_df["ranking_score"] = (
    60 + 30 * np.power(scaled, 0.6)
).round(2)

In [144]:
scaled = MinMaxScaler((65, 90))

candidate_df["ranking_score"] = scaled.fit_transform(
    candidate_df[["raw_score"]]
).round(2)

In [176]:
# ============================================================
# Skill Intelligence Engine
# ============================================================

SKILL_CATEGORIES = {

    "Retrieval & Search": [
        "faiss","bm25","elasticsearch","opensearch",
        "retrieval","vector search","pinecone",
        "milvus","qdrant","weaviate"
    ],

    "NLP & LLM": [
        "nlp","bert","transformers",
        "sentence transformers","llm",
        "huggingface","spacy","langchain"
    ],

    "Machine Learning": [
        "tensorflow","pytorch","xgboost",
        "lightgbm","catboost",
        "scikit-learn","machine learning"
    ],

    "Recommendation Systems": [
        "recommendation",
        "ranking",
        "collaborative filtering",
        "recommender"
    ],

    "Data Engineering": [
        "spark","hadoop",
        "airflow","kafka",
        "bigquery","snowflake"
    ],

    "Cloud & Deployment": [
        "docker","kubernetes",
        "aws","gcp","azure",
        "mlflow"
    ]
}

In [177]:
def detect_specialization(skills):

    skills = [s.lower() for s in skills]

    scores = {}

    for category, keywords in SKILL_CATEGORIES.items():

        score = 0

        for kw in keywords:

            for skill in skills:

                if kw in skill:
                    score += 1

        scores[category] = score

    best = max(scores, key=scores.get)

    if scores[best] == 0:
        return "Software Engineering"

    return best

In [178]:
candidate_df["specialization"] = (
    candidate_df["top_skills"]
    .apply(detect_specialization)
)

In [179]:
def recruiter_summary(row):

    years = row["career_months"] / 12
    skills = ", ".join(row["top_skills"][:3])

    spec = row["specialization"]

    # ---------- Semantic ----------
    if row["semantic_alignment"] >= 0.85:
        fit = "excellent"
    elif row["semantic_alignment"] >= 0.78:
        fit = "strong"
    else:
        fit = "good"

    # ---------- Specialization ----------
    spec_reason = {
        "Retrieval & Search":
            "demonstrates strong search and retrieval expertise",

        "NLP & LLM":
            "has solid NLP and LLM experience",

        "Machine Learning":
            "has extensive machine learning expertise",

        "Recommendation Systems":
            "shows strong recommendation system experience",

        "Data Engineering":
            "has experience building large-scale data pipelines",

        "Cloud & Deployment":
            "has production ML deployment experience",

        "Software Engineering":
            "has a strong software engineering background"
    }

    return (
        f"{row['current_role'].title()} with {years:.1f} years of experience; "
        f"{spec_reason[spec]}. "
        f"Ranked highly due to {fit} semantic alignment and expertise in {skills}."
    )

In [180]:
candidate_df["reasoning"] = (
    candidate_df.apply(recruiter_summary, axis=1)
)

In [181]:
candidate_df[
[
    "rank",
    "candidate_id",
    "ranking_score",
    "reasoning"
]
].head(10)

,rank,candidate_id,ranking_score,reasoning
0,1,CAND_0096172,90.00,Nlp Engineer with 3.3 years of experience; dem...
1,2,CAND_0091909,89.90,Machine Learning Engineer with 4.2 years of ex...
2,3,CAND_0091534,89.85,Ai Engineer with 4.3 years of experience; has ...
3,4,CAND_0036184,89.32,Recommendation Systems Engineer with 4.3 years...
4,5,CAND_0020708,89.30,Search Engineer with 4.2 years of experience; ...
5,6,CAND_0075249,89.06,Applied Ml Engineer with 3.0 years of experien...
6,7,CAND_0092706,88.86,Ai Research Engineer with 3.7 years of experie...
7,8,CAND_0089552,88.68,Machine Learning Engineer with 4.2 years of ex...
8,9,CAND_0078042,88.66,Applied Ml Engineer with 3.2 years of experien...
9,10,CAND_0069773,88.54,Ai Research Engineer with 4.2 years of experie...


In [182]:
TOP_K = 100

submission = candidate_df.head(TOP_K).copy()

submission["rank"] = range(1, TOP_K + 1)
submission["score"] = submission["ranking_score"].round(2)

submission = submission[
    [
        "candidate_id",
        "rank",
        "score",
        "reasoning"
    ]
]

submission.to_csv("submission.csv", index=False)

print("✅ submission.csv created successfully!")
print(submission.head())

✅ submission.csv created successfully!
   candidate_id  rank  score                                          reasoning
0  CAND_0096172     1  90.00  Nlp Engineer with 3.3 years of experience; dem...
1  CAND_0091909     2  89.90  Machine Learning Engineer with 4.2 years of ex...
2  CAND_0091534     3  89.85  Ai Engineer with 4.3 years of experience; has ...
3  CAND_0036184     4  89.32  Recommendation Systems Engineer with 4.3 years...
4  CAND_0020708     5  89.30  Search Engineer with 4.2 years of experience; ...
